#Ingest Sprints  file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns
-        Source File
-        Ingestion Timestamp
3. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/02.bronze-helper

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/sprints'
table_name = f"{catalog_name}.{bronze_schema}.sprints"

### Step 1 - Read the file using spark dataframe reader API

In [0]:
#Define the schema
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, DoubleType

sprints_schema = StructType([
    StructField("date", DateType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", DoubleType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
])

In [0]:
sprints_schema

In [0]:

sprints_df = (
     spark.read
     .format("json")
     .schema(sprints_schema)
     .option("multiLine","true")
     .option("mode","FAILFAST")
     .load(source_file)
)


In [0]:
display(sprints_df)

### Step 2 - Add metadata columns

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

###Step3 - Write to bronze delta table

In [0]:
write_to_bronze(sprints_final_df, table_name,v_batch_id)

In [0]:
display(spark.table(table_name))

In [0]:
%sql
select season,count(*)
from formula1_catalog.bronze.sprints
group by season
order by season;